In [2]:
import pandas as pd
from numpy.f2py.crackfortran import verbose
from sklearn import pipeline

data = pd.read_csv("../data/raw/banking_transactions.csv", sep=";")
data = data.drop(columns=['nameOrig','nameDest'])
print(data)

      step      type    amount oldbalanceOrg newbalanceOrig oldbalanceDest  \
0        1   PAYMENT   9839.64      170136.0      160296.36            0.0   
1        1   PAYMENT   1864.28       21249.0       19384.72            0.0   
2        1  TRANSFER    181.00         181.0            0.0            0.0   
3        1  CASH_OUT    181.00         181.0            0.0        21182.0   
4        1   PAYMENT  11668.14       41554.0       29885.86            0.0   
...    ...       ...       ...           ...            ...            ...   
9993     7   PAYMENT   6895.23        6026.0            0.0            0.0   
9994     7   PAYMENT    115.38      249688.0      249572.62            0.0   
9995     7   PAYMENT    466.73         881.0         414.27            0.0   
9996     7   PAYMENT   8239.66       11515.0        3275.34            0.0   
9997     7   PAYMENT   6063.34       31409.0       25345.66            0.0   

     newbalanceDest  isFraud  isFlaggedFraud  
0               

In [ ]:
import os
os.makedirs("../models", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)

In [49]:
money = ["oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]

for c in money:
    data[c] = pd.to_numeric(data[c], errors="coerce")
("encoder", OneHotEncoder(categories=[type]))
print(data[money].dtypes)          # phải là float64
print(data[money].isna().sum())    # số ô không đổi được thành số

oldbalanceOrg     float64
newbalanceOrig    float64
oldbalanceDest    float64
newbalanceDest    float64
dtype: object
oldbalanceOrg      67
newbalanceOrig     75
oldbalanceDest    284
newbalanceDest    354
dtype: int64


In [67]:
from data_profiling import ProfileReport
profile =ProfileReport(data,title='Profile Report',explorative=True)
profile.to_file("../reports/Report_bank.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 107.02it/s]


In [57]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target = 'isFraud'
x = data.drop(target,axis=1)
y = data[target]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
## cleaning data
# x_train, x_test: bảng đầu vào đã bỏ isFraud, nameOrig, nameDest, isFlaggedFraud
num_cols = x_train.select_dtypes("number").columns.tolist()
cat_cols = ["type"]

nom_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),   # bỏ missing_values=-1
    ("scaler", StandardScaler())
])
preprocessor = ColumnTransformer(transformers=[
("num_feature", num_transformer, ['step','amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest','isFlaggedFraud']),("nom_feature", nom_transformer, ['type']),
])

In [58]:
    from lazypredict.Supervised import LazyClassifier

clf = LazyClassifier(
    verbose=0,
    ignore_warnings=True,
    custom_metric=None)

models, predictions = clf.fit(x_train,x_test,y_train,y_test)

print(models)

                             Accuracy  Balanced Accuracy   ROC AUC  F1 Score  \
Model                                                                          
GaussianNB                     0.8170           0.907762  0.953409  0.891846   
BaggingClassifier              0.9955           0.811744  0.841450  0.995269   
DecisionTreeClassifier         0.9935           0.810736  0.810736  0.993598   
RandomForestClassifier         0.9965           0.781250  0.994755  0.996013   
AdaBoostClassifier             0.9955           0.749748  0.995290  0.994874   
ExtraTreesClassifier           0.9955           0.718750  0.931641  0.994625   
ExtraTreeClassifier            0.9940           0.717994  0.717994  0.993312   
NearestCentroid                0.8940           0.698589  0.918237  0.936807   
PassiveAggressiveClassifier    0.9935           0.624748  0.917181  0.991807   
CalibratedClassifierCV         0.9935           0.624748  0.933594  0.991807   
LinearDiscriminantAnalysis     0.9900   

In [59]:
from sklearn.svm import SVC
clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('model',SVC())
])
clf.fit(x_train,y_train)
y_predict = clf.predict(x_test)
for i,j in zip(y_predict,y_test):
    print('Predict value: {}. Actual value {}'.format(i,j))

Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict value: 0. Actual value 0
Predict va

In [62]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, fbeta_score

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
scoring = {
    "f2": make_scorer(fbeta_score, beta=2),
    "recall": "recall",
    "precision": "precision",
}

clf_svc = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", SVC(class_weight="balanced", random_state=42))
])

params = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", 0.01, 0.1],
    "model__kernel": ["rbf", "linear"],
}

grid_search = GridSearchCV(
    clf_svc, params,
    cv=cv, scoring=scoring, refit="f2",
    n_jobs=-1, verbose=2
)
grid_search.fit(x_train, y_train)

print("Tham số tốt nhất:", grid_search.best_params_)
i = grid_search.best_index_
print("F2        :", round(grid_search.cv_results_["mean_test_f2"][i], 3))
print("Recall    :", round(grid_search.cv_results_["mean_test_recall"][i], 3))
print("Precision :", round(grid_search.cv_results_["mean_test_precision"][i], 3))

Fitting 4 folds for each of 24 candidates, totalling 96 fits
Tham số tốt nhất: {'model__C': 100, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
F2        : 0.584
Recall    : 0.904
Precision : 0.242


In [69]:
from sklearn.metrics import confusion_matrix, classification_report, recall_score, precision_score, fbeta_score
import joblib

best_model = grid_search.best_estimator_
y_pred = best_model.predict(x_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"Gian lận thật: {tp+fn} | Bắt được: {tp} | Bỏ sót: {fn} | Báo nhầm: {fp}")
print(f"Recall    : {recall_score(y_test, y_pred):.3f}")
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"F2        : {fbeta_score(y_test, y_pred, beta=2):.3f}")
print(classification_report(y_test, y_pred, digits=3))
joblib.dump(best_model, "../models/fraud_svc.joblib")

Gian lận thật: 16 | Bắt được: 11 | Bỏ sót: 5 | Báo nhầm: 45
Recall    : 0.688
Precision : 0.196
F2        : 0.458
              precision    recall  f1-score   support

           0      0.997     0.977     0.987      1984
           1      0.196     0.688     0.306        16

    accuracy                          0.975      2000
   macro avg      0.597     0.832     0.646      2000
weighted avg      0.991     0.975     0.982      2000



['../models/fraud_svc.joblib']

In [70]:
import pandas as pd
from sklearn.metrics import classification_report

rep = pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T
rep.to_csv("../reports/evaluation_report.csv")
print(rep.round(3))

              precision  recall  f1-score   support
0                 0.997   0.977     0.987  1984.000
1                 0.196   0.688     0.306    16.000
accuracy          0.975   0.975     0.975     0.975
macro avg         0.597   0.832     0.646  2000.000
weighted avg      0.991   0.975     0.982  2000.000


In [72]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Bình thường", "Gian lận"]
)
plt.title("Confusion matrix - SVC")
plt.savefig("../reports/figures/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\ROG\AppData\Local\Temp\ipykernel_26580\3473529172.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
